In [3]:
import copy
import random
import re
import json
import numpy as np
import pandas as pd
import lightning.pytorch as pl
import torch
#from braindecode.models import Deep4Net
#from braindecode.models.util import to_dense_prediction_model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from torch import nn
import torch.nn as nn
import torch.nn.functional as F

#from configs.preprocess_config import preprocess_config
from loss_methods import ClipLoss, SigLipLoss


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class ResBlock1D(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(ch, ch, kernel_size=3, padding=1),
            nn.BatchNorm1d(ch),
            nn.ReLU(),
            nn.Conv1d(ch, ch, kernel_size=3, padding=1),
            nn.BatchNorm1d(ch),
        )

    def forward(self, x):
        return F.relu(x + self.net(x))


class ZXMLEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, 128, kernel_size=9, padding=4),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )

        self.blocks = nn.Sequential(
            ResBlock1D(128),
            ResBlock1D(128),
            # ResBlock1D(128),
            # ResBlock1D(128),
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),   # [B, 128, 1]
            nn.Flatten(),
            nn.Linear(128, out_dim)
        )

    def forward(self, x):
        # x: [B, L] where L is the feature length of z_xml / gps_z_xml
        x = x.unsqueeze(1)            # [B, 1, L]
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        return F.normalize(x, dim=-1)


class ZREncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )

        self.blocks = nn.Sequential(
            ResBlock1D(128),
            ResBlock1D(128),
            # ResBlock1D(128),
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(128, out_dim)
        )

    def forward(self, x):
        # x: [B, L] where L is the feature length of z_r
        x = x.unsqueeze(1)            # [B, 1, L]
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        return F.normalize(x, dim=-1)


class ProjectionHead(nn.Module):
    def __init__(
        self,
        input_dim=128,
        output_dim=128,
        dropout_rate=0.3,
        num_fc_layers=2,
        transpose=False,
    ):
        super().__init__()
        self.num_layers = num_fc_layers
        self.transpose = transpose
        layers = nn.ModuleList()

        # Input projection layer
        layers.append(nn.Linear(input_dim, output_dim))
        layers.append(nn.BatchNorm1d(output_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout_rate))

        # Hidden layers
        for _ in range(num_fc_layers - 2):
            layers.append(nn.Linear(output_dim, output_dim))
            layers.append(nn.BatchNorm1d(output_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))

        # Output projection layer
        layers.append(nn.Linear(output_dim, output_dim))

        self.layers = layers
        self.projection = nn.Sequential(*layers)

    def forward(self, x):
        for layer in self.layers:
            if isinstance(layer, nn.BatchNorm1d) and self.transpose:
                x = layer(x.transpose(1, 2)).transpose(1, 2)
            else:
                x = layer(x)
        return x

/home/silvija/CLIP/clipenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/silvija/CLIP/clipenv/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/silvija/CLIP/clipenv/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [4]:
def clip_loss(zr, xml, logit_scale):
    # zr, xml: [B, D]
    logit_scale = logit_scale.exp()
    logit_scale = torch.clamp(logit_scale, max=100)

    logits = logit_scale * zr @ xml.T
    labels = torch.arange(zr.size(0), device=zr.device)

    loss = (
        F.cross_entropy(logits, labels,label_smoothing=0.1) +
        F.cross_entropy(logits.T, labels, label_smoothing=0.1)
    ) / 2

    return loss, logits



In [5]:
class SceneClipModelPad(pl.LightningModule):
    def __init__(
        self,
        xml_model_emb_dim=128,
        zr_model_emb_dim=128,
        projected_emb_dim=64,
        zr_model_pretrained=False,
        zr_model_trainable=True,
        xml_model_pretrained=False,
        xml_model_trainable=True,
        dropout_rate=0.3,
        num_fc_layers=1,
        lr=1e-4,
        lr_frac_lm=0,
        lr_proj=3e-4,
        weight_decay=1e-6,
        n_chans=21,
        contrastive_loss_temperature=0.07,
        contrastive_loss_func="clip",
        **kwargs,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        self.lr_proj = lr_proj
        self.lr_lm = self.lr * lr_frac_lm
        self.weight_decay = weight_decay
        self.n_chans = n_chans
        self.contrastive_loss_temperature = contrastive_loss_temperature
        self.contrastive_loss_func = contrastive_loss_func
        self.zr_model_emb_dim = zr_model_emb_dim
        self.xml_model_emb_dim = xml_model_emb_dim

        self.zr_encoder = ZREncoder(zr_model_emb_dim)
        self.xml_encoder = ZXMLEncoder(xml_model_emb_dim)

        self.xml_projection = ProjectionHead(
            input_dim=self.xml_model_emb_dim,
            output_dim=projected_emb_dim,
            dropout_rate=dropout_rate,
            num_fc_layers=num_fc_layers,
            transpose=False,
        )

        self.zr_projection = ProjectionHead(
            input_dim=zr_model_emb_dim,
            output_dim=projected_emb_dim,
            dropout_rate=dropout_rate,
            num_fc_layers=num_fc_layers,
            transpose=False,
        )

        init_logit_scale = np.log(1 / 0.07)
        self.logit_scale = nn.Parameter(torch.ones([]) * init_logit_scale)

        # optional feature storage
        self.features_train = []
        self.features_valid = []

    def forward(self, batch):
        zr_batch, xml_batch = batch

        zr_features = self.zr_encoder(zr_batch)
        xml_features = self.xml_encoder(xml_batch)

        zr_features_proj = self.zr_projection(zr_features)
        xml_features_proj = self.xml_projection(xml_features)

        return (
            zr_features,
            zr_features_proj,
            xml_features,
            xml_features_proj,
        )

    def training_step(self, batch, batch_idx):
        (
            zr_features,
            zr_features_proj,
            xml_features,
            xml_features_proj,
        ) = self.forward(batch)

        self.features_train.append(zr_features_proj.detach())

        loss, logits = clip_loss(
            zr_features_proj,
            xml_features_proj,
            self.logit_scale
        )

        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        (
            zr_features,
            zr_features_proj,
            xml_features,
            xml_features_proj,
        ) = self.forward(batch)

        self.features_valid.append(zr_features_proj.detach())

        loss, logits = clip_loss(
            zr_features_proj,
            xml_features_proj,
            self.logit_scale
        )

        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def test_step(self, batch, batch_idx):
        (
            zr_features,
            zr_features_proj,
            xml_features,
            xml_features_proj,
        ) = self.forward(batch)

        loss, logits = clip_loss(
            zr_features_proj,
            xml_features_proj,
            self.logit_scale
        )

        self.log("test_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def on_train_epoch_end(self):
        self.features_train = []

    def on_validation_epoch_end(self):
        if len(self.features_valid) > 0:
            features_valid = torch.cat(self.features_valid).cpu()
        self.features_valid = []
        return None

    def configure_optimizers(self):
        params = list(self.named_parameters())
        print("params")
        print([n for n, p in params])

        def is_encoder(n):
            return "encoder" in n

        def is_projection(n):
            return "projection" in n

        def is_logit(n):
            return "logit_scale" in n

        grouped_parameters = [
            {"params": [p for n, p in params if is_encoder(n)], "lr": self.lr},
            {"params": [p for n, p in params if is_projection(n)], "lr": self.lr_proj},
            {"params": [p for n, p in params if is_logit(n)], "lr": 1e-5},
        ]

        optimizer = torch.optim.AdamW(
            grouped_parameters,
            lr=self.lr,
            weight_decay=self.weight_decay,
            betas=(0.9, 0.98),
            eps=1e-8
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.trainer.max_epochs - 1
        )

        return [optimizer], [scheduler]


def on_save_checkpoint(checkpoint):
    for key in list(checkpoint["state_dict"].keys()):
        if "text_encoder" in key:
            print("deleting ", key)
            del checkpoint["state_dict"][key]

In [6]:
xml_path = "/data/hafeez/gps_z_xml.npy"
xmldata = np.load(xml_path)
print("gps_z_xml shape:", xmldata.shape)
xmldata = xmldata.astype(np.float32)

gps_z_xml shape: (10000, 768)


In [7]:
zrdata = np.load("/data/hafeez/zrs1.npy").astype(np.float32)
print("Loaded:", zrdata.shape)

Loaded: (10000, 107)


In [9]:
print("XML shape:", xmldata.shape)
print("ZR shape:", zrdata.shape)

assert xmldata.shape[0] == zrdata.shape[0], "Mismatch in number of samples"

XML shape: (10000, 768)
ZR shape: (10000, 107)


In [10]:
SEQ_LEN=107
PAD_TO=112
assert zrdata.shape[-1] == SEQ_LEN
pad_right = PAD_TO - SEQ_LEN
z_pad = np.pad(zrdata, ((0, 0), (0, pad_right)), constant_values = 0)  
print("original:", zrdata.shape, "padded:", z_pad.shape)

original: (10000, 107) padded: (10000, 112)


In [11]:
def compute_norm_stats(X):
    #X = np.stack([x.squeeze(0) for x in list_X], axis=0)  # [N, D]
    Xn = X / np.max(X)
    m = np.mean(Xn, axis=0)
    s = np.std(Xn, axis=0) + 1e-8
    return np.max(X), m, s

In [12]:
class NormalizeTransform:
    def __init__(self, max_val, mean, std):
        self.max_val = max_val
        self.mean = torch.tensor(mean, dtype=torch.float32)
        self.std = torch.tensor(std, dtype=torch.float32)

    def __call__(self, x):
        x = x / self.max_val
        return (x - self.mean) / self.std

In [13]:
from torch.utils.data import Dataset
class PairedContrastiveDataset(Dataset):
    def __init__(self, list_A, list_B, transform_A=None, transform_B=None):
        assert len(list_A) == len(list_B)
        self.list_A = list_A
        self.list_B = list_B
        self.transform_A = transform_A
        self.transform_B = transform_B

    def __len__(self):
        return len(self.list_A)

    def __getitem__(self, idx):
        A = torch.tensor(self.list_A[idx], dtype=torch.float32).squeeze(0)
        B = torch.tensor(self.list_B[idx], dtype=torch.float32).squeeze(0)

        if self.transform_A:
            A = self.transform_A(A)
        if self.transform_B:
            B = self.transform_B(B)

        return A, B


In [14]:
def train_val_split_indices(n, val_ratio=0.2, seed=42):
    rng = np.random.default_rng(seed)
    indices = np.arange(n)
    rng.shuffle(indices)

    n_val = int(n * val_ratio)
    val_idx = indices[:n_val]
    train_idx = indices[n_val:]

    return train_idx, val_idx


In [16]:
N = len(zrdata)
train_idx, val_idx = train_val_split_indices(N, val_ratio=0.2)


In [17]:
train_A = [z_pad[i] for i in train_idx]
train_B = [xmldata[i] for i in train_idx]

val_A = [z_pad[i] for i in val_idx]
val_B = [xmldata[i] for i in val_idx]

In [18]:
AT_max, AT_mean, AT_std = compute_norm_stats(train_A)
BT_max, BT_mean, BT_std = compute_norm_stats(train_B)
AV_max, AV_mean, AV_std = compute_norm_stats(val_A)
BV_max, BV_mean, BV_std = compute_norm_stats(val_B)

In [19]:
transform_AT = NormalizeTransform(AT_max, AT_mean, AT_std)
transform_BT = NormalizeTransform(BT_max, BT_mean, BT_std)
transform_AV = NormalizeTransform(AV_max, AV_mean, AV_std)
transform_BV = NormalizeTransform(BV_max, BV_mean, BV_std)


In [21]:
lr = 1e-4
lr_proj = 3e-4
weight_decay = 1e-6
n_chans = 1
num_fc_layers = 3
lr_frac_lm = 0
projected_emb_dim = 128
xml_model_emb_dim = 128
zr_model_emb_dim = 128
num_epochs = 200

train_dataset = PairedContrastiveDataset(
    train_A,
    train_B,
    transform_A=transform_AT,
    transform_B=transform_BT,
)

val_dataset = PairedContrastiveDataset(
    val_A,
    val_B,
    transform_A=transform_AV,
    transform_B=transform_BV,
)

trainloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
)

valloader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
)

model = SceneClipModelPad(
    n_chans=n_chans,
    lr=lr,
    lr_proj=lr_proj,
    lr_frac_lm=lr_frac_lm,
    weight_decay=weight_decay,
    num_fc_layers=num_fc_layers,
    projected_emb_dim=projected_emb_dim,
    xml_model_emb_dim=xml_model_emb_dim,
    zr_model_emb_dim=zr_model_emb_dim,
)

trainer = pl.Trainer(
    max_epochs=num_epochs,
    accelerator="auto",
    devices="auto",
)

trainer.fit(
    model,
    train_dataloaders=trainloader,
    val_dataloaders=valloader,
)

Trainer will use only 1 of 8 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=8)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]

  | Name           | Type           | Params | Mode  | FLOPs
------------------------------------------------------------------
0 | zr_encoder     | ZREncoder      | 215 K  | train | 0    
1 | xml_encoder    | ZXMLEncoder    | 216 K  | train | 0    
2 | xml_projection | ProjectionHead | 50.0 K | train | 0    
3 | zr_projection  | ProjectionHead | 50.0 K | train | 0    
  | ot

params
['logit_scale', 'zr_encoder.stem.0.weight', 'zr_encoder.stem.0.bias', 'zr_encoder.stem.1.weight', 'zr_encoder.stem.1.bias', 'zr_encoder.blocks.0.net.0.weight', 'zr_encoder.blocks.0.net.0.bias', 'zr_encoder.blocks.0.net.1.weight', 'zr_encoder.blocks.0.net.1.bias', 'zr_encoder.blocks.0.net.3.weight', 'zr_encoder.blocks.0.net.3.bias', 'zr_encoder.blocks.0.net.4.weight', 'zr_encoder.blocks.0.net.4.bias', 'zr_encoder.blocks.1.net.0.weight', 'zr_encoder.blocks.1.net.0.bias', 'zr_encoder.blocks.1.net.1.weight', 'zr_encoder.blocks.1.net.1.bias', 'zr_encoder.blocks.1.net.3.weight', 'zr_encoder.blocks.1.net.3.bias', 'zr_encoder.blocks.1.net.4.weight', 'zr_encoder.blocks.1.net.4.bias', 'zr_encoder.head.2.weight', 'zr_encoder.head.2.bias', 'xml_encoder.stem.0.weight', 'xml_encoder.stem.0.bias', 'xml_encoder.stem.1.weight', 'xml_encoder.stem.1.bias', 'xml_encoder.blocks.0.net.0.weight', 'xml_encoder.blocks.0.net.0.bias', 'xml_encoder.blocks.0.net.1.weight', 'xml_encoder.blocks.0.net.1.bias',

/home/silvija/CLIP/clipenv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
/home/silvija/CLIP/clipenv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 63/63 [00:02<00:00, 22.18it/s, v_num=4]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 63/63 [00:01<00:00, 35.80it/s, v_num=4, val_loss=6.580, train_loss=47.00]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2: 100%|██████████| 63/63 [00:02<00:00, 30.87it/s, v_num=4, val_loss=9.280, train_loss=23.80]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3: 100%|██████████| 63/63 [00:02<00:00, 30.87it/s, v_num=4, val_loss=7.490, train_loss=15.10]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 4: 100%|██████████| 63/63 [00:02<00:00, 30.13it/s, v_num=4, val_loss=6.590, train_loss=10.60]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 5: 100%|██████████| 63/63 [00:00<00:00, 66.18it/s, v_num=4, val_loss

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 199: 100%|██████████| 63/63 [00:02<00:00, 31.48it/s, v_num=4, val_loss=4.230, train_loss=3.720]


In [24]:
import os
import numpy as np
import torch

ckpt_path = "/home/silvija/saiteja/New_Zxml/lightning_logs/version_4/checkpoints/epoch=199-step=12600.ckpt"
model = SceneClipModelPad.load_from_checkpoint(ckpt_path)
model = model.to(device)
model.eval()

# Full-data normalization, following your professor's pipeline style
A_max, A_mean, A_std = compute_norm_stats(z_pad)
B_max, B_mean, B_std = compute_norm_stats(xmldata)

transform_A = NormalizeTransform(A_max, A_mean, A_std)
transform_B = NormalizeTransform(B_max, B_mean, B_std)

full_dataset = PairedContrastiveDataset(
    z_pad,
    xmldata,
    transform_A=transform_A,
    transform_B=transform_B,
)

full_loader = torch.utils.data.DataLoader(
    full_dataset,
    batch_size=128,
    shuffle=False,
)

all_zr_proj = []
all_xml_proj = []

with torch.no_grad():
    for batch in full_loader:
        zr_batch, xml_batch = [x.to(device) for x in batch]

        zr_features = model.zr_encoder(zr_batch)
        xml_features = model.xml_encoder(xml_batch)

        zr_proj = model.zr_projection(zr_features)
        xml_proj = model.xml_projection(xml_features)

        all_zr_proj.append(zr_proj.cpu().numpy())
        all_xml_proj.append(xml_proj.cpu().numpy())

all_zr_proj = np.concatenate(all_zr_proj, axis=0)
all_xml_proj = np.concatenate(all_xml_proj, axis=0)

print("zr projected shape :", all_zr_proj.shape)
print("xml projected shape:", all_xml_proj.shape)

out_dir = "/data/silvija/diffusion_results"
os.makedirs(out_dir, exist_ok=True)

np.save(os.path.join(out_dir, "zr_features_proj_clip_gps.npy"), all_zr_proj)
np.save(os.path.join(out_dir, "gps_xml_features_proj.npy"), all_xml_proj)

print("Saved:")
print(os.path.join(out_dir, "zr_features_proj_clip_gps.npy"))
print(os.path.join(out_dir, "gps_xml_features_proj.npy"))

zr projected shape : (10000, 128)
xml projected shape: (10000, 128)
Saved:
/data/silvija/diffusion_results/zr_features_proj_clip_gps.npy
/data/silvija/diffusion_results/gps_xml_features_proj.npy


In [29]:
#model = SceneClipModelPad.load_from_checkpoint("/data/silvija/CLIPcheckpoints/epoch=199-step=25000.ckpt")
import time
import os
os.environ['CUDA_VISIBLE_DEVICES']='0,1,2,3,4,5,6,7'
model = SceneClipModelPad.load_from_checkpoint("/home/silvija/saiteja/New_Zxml/lightning_logs/version_4/checkpoints/epoch=199-step=12600.ckpt")
_, test_idx = train_val_split_indices(N, val_ratio=0.3)
test_A = [z_pad[i] for i in test_idx]
test_B = [xmldata[i] for i in test_idx]
AT_max, AT_mean, AT_std = compute_norm_stats(test_A)
BT_max, BT_mean, BT_std = compute_norm_stats(test_B)

transform_AT = NormalizeTransform(AT_max, AT_mean, AT_std)
transform_BT = NormalizeTransform(BT_max, BT_mean, BT_std)

test_dataset = PairedContrastiveDataset(test_A, test_B, transform_A=transform_AT, transform_B=transform_BT,)

testloader = torch.utils.data.DataLoader(
                    test_dataset,
                    batch_size=1,
                    shuffle=True)

In [30]:
test_dataset[0][1].shape

torch.Size([768])

In [25]:
#model = SceneClipModelPad.load_from_checkpoint("/data/silvija/CLIPcheckpoints/epoch=199-step=25000.ckpt")
import time
import os
os.environ['CUDA_VISIBLE_DEVICES']='0,1,2,3,4,5,6,7'
model = SceneClipModelPad.load_from_checkpoint("/home/silvija/saiteja/New_Zxml/lightning_logs/version_4/checkpoints/epoch=199-step=12600.ckpt")
_, test_idx = train_val_split_indices(N, val_ratio=0.3)
test_A = [z_pad[i] for i in test_idx]
test_B = [xmldata[i] for i in test_idx]
AT_max, AT_mean, AT_std = compute_norm_stats(test_A)
BT_max, BT_mean, BT_std = compute_norm_stats(test_B)

transform_AT = NormalizeTransform(AT_max, AT_mean, AT_std)
transform_BT = NormalizeTransform(BT_max, BT_mean, BT_std)

test_dataset = PairedContrastiveDataset(test_A, test_B, transform_A=transform_AT, transform_B=transform_BT,)

testloader = torch.utils.data.DataLoader(
                    test_dataset,
                    batch_size=1,
                    shuffle=True)

In [26]:
test_dataset[0][1].shape

torch.Size([768])

In [27]:
import time
import os
import torch

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"

ckpt_path = "/home/silvija/saiteja/New_Zxml/lightning_logs/version_4/checkpoints/epoch=199-step=12600.ckpt"
model = SceneClipModelPad.load_from_checkpoint(ckpt_path)

_, test_idx = train_val_split_indices(N, val_ratio=0.3)

test_A = [z_pad[i] for i in test_idx]
test_B = [xmldata[i] for i in test_idx]

AT_max, AT_mean, AT_std = compute_norm_stats(test_A)
BT_max, BT_mean, BT_std = compute_norm_stats(test_B)

transform_AT = NormalizeTransform(AT_max, AT_mean, AT_std)
transform_BT = NormalizeTransform(BT_max, BT_mean, BT_std)

test_dataset = PairedContrastiveDataset(
    test_A,
    test_B,
    transform_A=transform_AT,
    transform_B=transform_BT,
)

testloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

model = torch.nn.DataParallel(model)
model = model.to("cuda")
model.eval()

xml_context = []

start_time = time.time()
with torch.no_grad():
    for data in testloader:
        _, x = data
        x = x.cuda()
        x_proj = model.module.xml_projection(model.module.xml_encoder(x))
        xml_context.append(x_proj.cpu())

elapsed_time = time.time() - start_time
print(f"Elapsed time: {elapsed_time:.6f} seconds")

xml_context = torch.cat(xml_context, dim=0)
print("xml_context shape:", xml_context.shape)

Elapsed time: 0.159297 seconds
xml_context shape: torch.Size([3000, 128])


In [28]:

import time
import os
os.environ['CUDA_VISIBLE_DEVICES']='0,1,2,3,4,5,6,7'
model = SceneClipModelPad.load_from_checkpoint("/home/silvija/saiteja/New_Zxml/lightning_logs/version_4/checkpoints/epoch=199-step=12600.ckpt")
_, test_idx = train_val_split_indices(N, val_ratio=0.3)
test_A = [z_pad[i] for i in test_idx]
test_B = [xmldata[i] for i in test_idx]
AT_max, AT_mean, AT_std = compute_norm_stats(test_A)
BT_max, BT_mean, BT_std = compute_norm_stats(test_B)

transform_AT = NormalizeTransform(AT_max, AT_mean, AT_std)
transform_BT = NormalizeTransform(BT_max, BT_mean, BT_std)

test_dataset = PairedContrastiveDataset(test_A, test_B, transform_A=transform_AT, transform_B=transform_BT,)

testloader = torch.utils.data.DataLoader(
                    test_dataset,
                    batch_size=1,
                    shuffle=True)
# disable randomness, dropout, etc...
model = torch.nn.DataParallel(model)
model.to('cuda')
model.eval()



# predict with the model
xml_context=[]
with torch.no_grad():
    for data in testloader:
        start_time = time.time()
        _,x=data
        x=x.cuda()
        xml_context.append(model.module.xml_projection(model.module.xml_encoder(x)))
        end_time = time.time()
        elapsed_time = time.time() - start_time
print(f"Elapsed time: {elapsed_time:.6f} seconds")
  

Elapsed time: 0.001146 seconds
